In [20]:
# !pip install pandas matplotlib seaborn scikit-learn openai python-dotenv

None
None


In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()  # This loads variables from .env into os.environ


# Note: The base_url varies by region. The following example uses the base_url for the Singapore region.
# - Singapore: https://dashscope-intl.aliyuncs.com/compatible-mode/v1
# - US (Virginia): https://dashscope-us.aliyuncs.com/compatible-mode/v1
# - China (Beijing): https://dashscope.aliyuncs.com/compatible-mode/v1
# - China (Hong Kong): https://cn-hongkong.dashscope.aliyuncs.com/compatible-mode/v1
# - Germany (Frankfurt): https://{WorkspaceId}.eu-central-1.maas.aliyuncs.com/compatible-mode/v1. Replace {WorkspaceId} with your workspace ID.
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"), 
    base_url=os.getenv("DASHSCOPE_API_URL")
)
completion = client.chat.completions.create(
    model="qwen-max",  # Changed from "qwen3.6-plus"
    messages=[{"role": "user", "content": "Who are you?"}]
)
print(completion.choices[0].message.content)

I'm Qwen, a large language model created by Alibaba Cloud. I'm here to help you with a wide range of tasks, from answering questions and writing stories to expressing opinions and providing information. How can I assist you today?


# RETRIEVE

In [3]:
mem_stream = "data/topic1_construction_welding.txt"

In [4]:
import re
from datetime import datetime
from typing import List, Dict, Tuple
import numpy as np

# ============================================================================
# PHASE 1: Parse and Group Memories from Data File
# ============================================================================

def parse_conversation_file(file_path: str) -> List[Dict]:
    """
    Parse conversation file into timestamped exchanges.
    Returns list of dicts with: timestamp, speaker, role, message
    """
    exchanges = []
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Format: HH:MM:SS - Name (Role): Message
            match = re.match(r'^(\d{2}:\d{2}:\d{2})\s*-\s*(.+?)\s*\((.+?)\)\s*:\s*(.+)$', line)
            if match:
                timestamp, speaker, role, message = match.groups()
                exchanges.append({
                    'timestamp': timestamp,
                    'speaker': speaker.strip(),
                    'role': role.strip(),
                    'message': message.strip()
                })
    return exchanges

def group_memories(exchanges: List[Dict], time_gap_threshold: int = 5) -> List[Dict]:
    """
    Group related exchanges into coherent memories based on:
    - Time proximity (if gap > time_gap_threshold minutes, new memory)
    - Topic similarity (simple heuristic: if no speakers overlap in dialogue, might be new topic)
    
    Returns list of memory groups with: timestamp_start, timestamp_end, speakers, full_text, exchange_indices
    """
    if not exchanges:
        return []
    
    memories = []
    current_group = [exchanges[0]]
    
    for i in range(1, len(exchanges)):
        prev_time = datetime.strptime(exchanges[i-1]['timestamp'], '%H:%M:%S')
        curr_time = datetime.strptime(exchanges[i]['timestamp'], '%H:%M:%S')
        time_diff_seconds = (curr_time - prev_time).total_seconds()
        time_diff_minutes = time_diff_seconds / 60
        
        # If time gap exceeds threshold, start new memory
        if time_diff_minutes > time_gap_threshold:
            # Save current group as a memory
            memories.append(_group_to_memory(current_group))
            current_group = [exchanges[i]]
        else:
            current_group.append(exchanges[i])
    
    # Don't forget the last group
    if current_group:
        memories.append(_group_to_memory(current_group))
    
    return memories

def _group_to_memory(group: List[Dict]) -> Dict:
    """Convert a group of exchanges into a memory dict"""
    timestamp_start = group[0]['timestamp']
    timestamp_end = group[-1]['timestamp']
    speakers = list(set(ex['speaker'] for ex in group))
    full_text = '\n'.join([f"{ex['timestamp']} - {ex['speaker']} ({ex['role']}): {ex['message']}" for ex in group])
    
    return {
        'timestamp_start': timestamp_start,
        'timestamp_end': timestamp_end,
        'speakers': speakers,
        'full_text': full_text,
        'exchange_count': len(group)
    }

# Parse the file
data_file = 'data/topic1_construction_welding.txt'
exchanges = parse_conversation_file(data_file)
print(f"Total exchanges parsed: {len(exchanges)}")
print("\nFirst 3 exchanges:")
for ex in exchanges[:3]:
    print(f"  {ex['timestamp']} - {ex['speaker']} ({ex['role']}): {ex['message'][:60]}...")

# Group into memories
memories = group_memories(exchanges, time_gap_threshold=5)
print(f"\n\nTotal memories grouped: {len(memories)}")
print("\nMemory groups:")
for i, mem in enumerate(memories):
    print(f"  Memory {i+1}: {mem['timestamp_start']}-{mem['timestamp_end']} | Speakers: {mem['speakers']} | Exchanges: {mem['exchange_count']}")


Total exchanges parsed: 21

First 3 exchanges:
  07:02:15 - Leo (Entry-Level Welder): Morning everyone. I'm assigned to welding station 3 today....
  07:03:10 - Mack (Senior Welder): Heads up Leo, station 3 always overheats on Tuesdays after l...
  07:03:45 - Leo (Entry-Level Welder): Oh, I didn't see that in the manual. What should I do when i...


Total memories grouped: 5

Memory groups:
  Memory 1: 07:02:15-07:07:30 | Speakers: ['Sarah', 'Mack', 'Leo'] | Exchanges: 8
  Memory 2: 07:15:00-07:18:15 | Speakers: ['Leo', 'Mack'] | Exchanges: 5
  Memory 3: 07:30:00-07:30:00 | Speakers: ['Leo'] | Exchanges: 1
  Memory 4: 08:45:12-08:48:00 | Speakers: ['John', 'Sarah', 'Mack'] | Exchanges: 4
  Memory 5: 11:30:00-11:32:00 | Speakers: ['Leo', 'Mack'] | Exchanges: 3


In [5]:
# ============================================================================
# PHASE 2: Score Each Memory Using qwen-max Model
# ============================================================================

def build_scoring_prompt(memory: Dict, current_time: str, current_situation: str) -> str:
    """Build the scoring prompt for a memory"""
    prompt = f"""Given the current situation and a specific memory from the factory floor, rate the memory on a scale of 1 to 10 for four distinct criteria: Recency, Importance, Relevance, and Experience.

Current Time: {current_time}
Current Situation/Query: {current_situation}

Memory Timestamp: {memory['timestamp_start']}
Memory Record:
{memory['full_text']}

Scoring Criteria:
1. Recency: How recent is this memory relative to the Current Time? (1 = very old/outdated, 10 = just happened)
2. Importance: How critical is this operational knowledge? (1 = mundane noise, venting, or personal complaints, 10 = highly critical unwritten tribal wisdom, safety hazards, or SOP contradictions)
3. Relevance: How closely does this memory relate to the Current Situation? (1 = completely unrelated, 10 = directly answers or provides context to the situation)
4. Experience: How much expertise/seniority does the speaker have in this operational area? (1 = novice/intern, 10 = highly experienced expert)

Output your response in the following EXACT format (one score per line):
Recency: [Score]
Importance: [Score]
Relevance: [Score]
Experience: [Score]"""
    return prompt

def extract_scores_from_response(response_text: str) -> Dict or None:
    """Extract the four scores from LLM response"""
    lines = response_text.strip().split('\n')
    scores = {}
    
    for line in lines:
        if ':' in line:
            key, value = line.split(':', 1)
            key = key.strip().lower()
            try:
                value = int(value.strip())
                if key in ['recency', 'importance', 'relevance', 'experience']:
                    scores[key] = value
            except ValueError:
                pass
    
    # Check if all four scores are present
    if len(scores) == 4 and all(k in scores for k in ['recency', 'importance', 'relevance', 'experience']):
        return scores
    return None

def score_memories_batch(memories: List[Dict], client, current_time: str, current_situation: str) -> List[Dict]:
    """Score all memories using qwen-max model"""
    scored_memories = []
    
    for i, memory in enumerate(memories):
        print(f"Scoring memory {i+1}/{len(memories)}...")
        prompt = build_scoring_prompt(memory, current_time, current_situation)
        
        try:
            completion = client.chat.completions.create(
                model="qwen-max",
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = completion.choices[0].message.content
            scores = extract_scores_from_response(response_text)
            
            if scores:
                memory['scores'] = scores
                memory['raw_response'] = response_text
                scored_memories.append(memory)
                print(f"  ✓ Scores: R={scores['recency']} I={scores['importance']} Rel={scores['relevance']} E={scores['experience']}")
            else:
                print(f"  ✗ Failed to parse scores from response")
        except Exception as e:
            print(f"  ✗ API Error: {str(e)}")
    
    return scored_memories

# Determine current time (latest timestamp in file)
current_time = memories[-1]['timestamp_end'] if memories else datetime.now().strftime('%H:%M:%S')
current_situation = "Filter factory floor conversations to identify meaningful operational/safety knowledge while removing noise (venting, personal complaints)"

print("="*80)
print("PHASE 2: Scoring Memories with qwen-max Model")
print("="*80)
print(f"Current Time: {current_time}")
print(f"Current Situation: {current_situation}\n")

# Score all memories
scored_memories = score_memories_batch(memories, client, current_time, current_situation)
print(f"\nSuccessfully scored {len(scored_memories)}/{len(memories)} memories")


PHASE 2: Scoring Memories with qwen-max Model
Current Time: 11:32:00
Current Situation: Filter factory floor conversations to identify meaningful operational/safety knowledge while removing noise (venting, personal complaints)

Scoring memory 1/5...
  ✓ Scores: R=4 I=8 Rel=9 E=7
Scoring memory 2/5...
  ✓ Scores: R=4 I=6 Rel=7 E=8
Scoring memory 3/5...
  ✓ Scores: R=4 I=2 Rel=3 E=1
Scoring memory 4/5...
  ✓ Scores: R=7 I=5 Rel=8 E=6
Scoring memory 5/5...
  ✓ Scores: R=10 I=1 Rel=1 E=8

Successfully scored 5/5 memories


In [7]:
# ============================================================================
# PHASE 3: Normalize Scores and Filter Memories
# ============================================================================

def normalize_scores_minmax(scored_memories: List[Dict]) -> List[Dict]:
    """Apply min-max normalization to each criterion across all memories"""
    if not scored_memories:
        return []
    
    # Collect all raw scores for each criterion
    criteria = ['recency', 'importance', 'relevance', 'experience']
    raw_scores = {crit: [] for crit in criteria}
    
    for memory in scored_memories:
        for crit in criteria:
            raw_scores[crit].append(memory['scores'][crit])
    
    # Calculate min/max for each criterion
    min_max = {}
    for crit in criteria:
        min_val = min(raw_scores[crit])
        max_val = max(raw_scores[crit])
        min_max[crit] = (min_val, max_val)
    
    # Normalize and add normalized scores
    normalized_memories = []
    for memory in scored_memories:
        normalized = {}
        final_score = 0
        
        for crit in criteria:
            raw = memory['scores'][crit]
            min_val, max_val = min_max[crit]
            
            # Handle case where all values are the same
            if max_val == min_val:
                norm = 0.5  # Default to midpoint
            else:
                norm = (raw - min_val) / (max_val - min_val)
            
            normalized[f'{crit}_raw'] = raw
            normalized[f'{crit}_norm'] = norm
            final_score += norm
        
        memory['normalized_scores'] = normalized
        memory['final_score'] = final_score  # Max 4.0 after normalization
        normalized_memories.append(memory)
    
    return normalized_memories

def filter_and_sort_memories(normalized_memories: List[Dict], threshold: float = 3.5) -> List[Dict]:
    """Filter memories by score threshold and sort by score descending"""
    filtered = [mem for mem in normalized_memories if mem['final_score'] >= threshold]
    sorted_memories = sorted(filtered, key=lambda x: x['final_score'], reverse=True)
    return sorted_memories

print("="*80)
print("PHASE 3: Score Normalization & Filtering")
print("="*80)

# Normalize scores
normalized_memories = normalize_scores_minmax(scored_memories)
print(f"Normalized {len(normalized_memories)} memories")

# Show raw vs normalized scores for first memory
if normalized_memories:
    mem = normalized_memories[0]
    print(f"\nExample - Memory 1 Score Transformation:")
    print(f"  Raw Scores: R={mem['scores']['recency']} I={mem['scores']['importance']} Rel={mem['scores']['relevance']} E={mem['scores']['experience']}")
    norm = mem['normalized_scores']
    print(f"  Normalized: R={norm['recency_norm']:.3f} I={norm['importance_norm']:.3f} Rel={norm['relevance_norm']:.3f} E={norm['experience_norm']:.3f}")
    print(f"  Final Score: {mem['final_score']:.3f} / 4.0")

# Filter by threshold
threshold = 3.5
filtered_memories = filter_and_sort_memories(normalized_memories, threshold=threshold)
print(f"\nFiltered memories (score >= {threshold}): {len(filtered_memories)} / {len(normalized_memories)}")

print(f"\nTop 5 Memories by Score:")
for i, mem in enumerate(filtered_memories[:5]):
    print(f"  {i+1}. Score: {mem['final_score']:.3f} | Speakers: {mem['speakers']} | Time: {mem['timestamp_start']}-{mem['timestamp_end']}")
    print(f"     Text: {mem['full_text'][:80]}...")


PHASE 3: Score Normalization & Filtering
Normalized 5 memories

Example - Memory 1 Score Transformation:
  Raw Scores: R=4 I=8 Rel=9 E=7
  Normalized: R=0.000 I=1.000 Rel=1.000 E=0.857
  Final Score: 2.857 / 4.0

Filtered memories (score >= 3.5): 0 / 5

Top 5 Memories by Score:


In [8]:
# Show score distribution for all normalized memories
print("\nAll Normalized Memories (sorted by score):")
all_sorted = sorted(normalized_memories, key=lambda x: x['final_score'], reverse=True)
for i, mem in enumerate(all_sorted):
    norm = mem['normalized_scores']
    print(f"  {i+1}. Score: {mem['final_score']:.3f} | R={norm['recency_norm']:.2f} I={norm['importance_norm']:.2f} Rel={norm['relevance_norm']:.2f} E={norm['experience_norm']:.2f}")
    print(f"     {mem['full_text'][:70]}...")

# Re-filter with a lower threshold that captures meaningful memories
print(f"\nAdjusting threshold from 3.5 to 2.0 to capture meaningful memories...")
filtered_memories = filter_and_sort_memories(normalized_memories, threshold=2.0)


All Normalized Memories (sorted by score):
  1. Score: 2.857 | R=0.00 I=1.00 Rel=1.00 E=0.86
     07:02:15 - Leo (Entry-Level Welder): Morning everyone. I'm assigned to...
  2. Score: 2.661 | R=0.50 I=0.57 Rel=0.88 E=0.71
     08:45:12 - John (Intern): Has anyone seen the blueprint for the HVAC l...
  3. Score: 2.464 | R=0.00 I=0.71 Rel=0.75 E=1.00
     07:15:00 - Mack (Senior Welder): Man, the safety engineers who write t...
  4. Score: 2.000 | R=1.00 I=0.00 Rel=0.00 E=1.00
     11:30:00 - Mack (Senior Welder): I swear, if the cafeteria serves that...
  5. Score: 0.393 | R=0.00 I=0.14 Rel=0.25 E=0.00
     07:30:00 - Leo (Entry-Level Welder): Wiping it worked. Thanks....

Adjusting threshold from 3.5 to 2.0 to capture meaningful memories...


In [9]:
# ============================================================================
# PHASE 4: Context Window Fitting & Memory Selection
# ============================================================================

def estimate_tokens(text: str, tokens_per_word: float = 0.25) -> int:
    """Estimate token count using simple heuristic: ~4 words per token (0.25 tokens per word)"""
    word_count = len(text.split())
    return int(word_count * (1 / tokens_per_word))  # Conservative: 1 token per 4 words

def select_memories_for_context(filtered_memories: List[Dict], 
                                 context_window_tokens: int = 6000,
                                 system_prompt_tokens: int = 1500) -> Tuple[List[Dict], int]:
    """
    Select top-ranked memories that fit within context window.
    Returns: (selected_memories, tokens_used)
    """
    available_tokens = context_window_tokens - system_prompt_tokens
    selected = []
    tokens_used = 0
    
    for memory in filtered_memories:
        mem_tokens = estimate_tokens(memory['full_text'])
        
        if tokens_used + mem_tokens <= available_tokens:
            selected.append(memory)
            tokens_used += mem_tokens
        else:
            # Memory doesn't fit, skip it and continue
            pass
    
    return selected, tokens_used

print("="*80)
print("PHASE 4: Context Window Fitting & Memory Selection")
print("="*80)

# Estimate tokens for each filtered memory
for mem in filtered_memories[:5]:
    tokens = estimate_tokens(mem['full_text'])
    mem['estimated_tokens'] = tokens
    print(f"Memory: {mem['timestamp_start']}-{mem['timestamp_end']} | Tokens: {tokens} | Score: {mem['final_score']:.3f}")

# Select memories that fit within context window (6000 tokens available)
context_window = 6000
system_prompt_tokens = 1500  # Reserved for agent description, worker input, etc.
selected_memories, tokens_used = select_memories_for_context(
    filtered_memories, 
    context_window_tokens=context_window,
    system_prompt_tokens=system_prompt_tokens
)

print(f"\nSelected {len(selected_memories)} memories for context")
print(f"Context used: {tokens_used} tokens / {context_window - system_prompt_tokens} available")
print(f"Remaining tokens: {(context_window - system_prompt_tokens) - tokens_used}")

print(f"\nSelected Memories for Final Prompt:")
for i, mem in enumerate(selected_memories):
    print(f"  {i+1}. Score: {mem['final_score']:.3f} | Tokens: {mem['estimated_tokens']} | Speakers: {mem['speakers']}")
    print(f"     {mem['full_text'][:100]}...")


PHASE 4: Context Window Fitting & Memory Selection
Memory: 07:02:15-07:07:30 | Tokens: 612 | Score: 2.857
Memory: 08:45:12-08:48:00 | Tokens: 224 | Score: 2.661
Memory: 07:15:00-07:18:15 | Tokens: 348 | Score: 2.464
Memory: 11:30:00-11:32:00 | Tokens: 168 | Score: 2.000

Selected 4 memories for context
Context used: 1352 tokens / 4500 available
Remaining tokens: 3148

Selected Memories for Final Prompt:
  1. Score: 2.857 | Tokens: 612 | Speakers: ['Sarah', 'Mack', 'Leo']
     07:02:15 - Leo (Entry-Level Welder): Morning everyone. I'm assigned to welding station 3 today.
07:0...
  2. Score: 2.661 | Tokens: 224 | Speakers: ['John', 'Sarah', 'Mack']
     08:45:12 - John (Intern): Has anyone seen the blueprint for the HVAC layout?
08:46:05 - Mack (Senior...
  3. Score: 2.464 | Tokens: 348 | Speakers: ['Leo', 'Mack']
     07:15:00 - Mack (Senior Welder): Man, the safety engineers who write these manuals have never actual...
  4. Score: 2.000 | Tokens: 168 | Speakers: ['Leo', 'Mack']
     11

In [10]:
# ============================================================================
# PHASE 5: Build Final Prompt & Generate Agent Response with qwen3.5-flash
# ============================================================================

def build_final_prompt(selected_memories: List[Dict], 
                      current_time: str,
                      worker_input: str) -> str:
    """Build the final prompt for the Minder Agent"""
    
    agent_description = """[Agent's Summary Description]
Name: Minder Agent
Core Directive: Assist factory floor workers with Standard Operating Procedures (SOPs) and safely acquire, verify, and document unwritten tribal knowledge. Ignore venting, noise, and jokes."""
    
    # Format selected memories
    memories_text = "Summary of relevant context from the Agent's memory:"
    for i, mem in enumerate(selected_memories):
        memories_text += f"\n\nMemory {i+1} (Score: {mem['final_score']:.3f}):\n{mem['full_text']}"
    
    final_prompt = f"""{agent_description}

Current Time: {current_time}

Worker Input (Observation): {worker_input}

{memories_text}

Given the worker's input and the relevant context retrieved from memory, how should the Minder Agent respond? If there is a contradiction in the memory context, ask the worker for clarification or escalate to a manager."""
    
    return final_prompt

def get_agent_response(client, final_prompt: str, model: str = "qwen3.5-flash") -> str:
    """Get agent response from qwen3.5-flash"""
    try:
        completion = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": final_prompt}]
        )
        response = completion.choices[0].message.content
        return response
    except Exception as e:
        return f"Error generating response: {str(e)}"

print("="*80)
print("PHASE 5: Final Prompt Building & Agent Response Generation")
print("="*80)

# Define a test worker input (example)
test_worker_input = "I'm working on welding station 3 today. What should I do if the station overheats?"

print(f"\nTest Worker Input: {test_worker_input}\n")

# Build final prompt
final_prompt = build_final_prompt(selected_memories, current_time, test_worker_input)
print("Final Prompt Built")
print(f"Prompt length: {len(final_prompt)} characters")
print(f"Prompt tokens (estimated): {estimate_tokens(final_prompt)}\n")

print("-"*80)
print("FINAL AGENT RESPONSE (using qwen3.5-flash):")
print("-"*80)

# Get agent response
agent_response = get_agent_response(client, final_prompt, model="qwen3.5-flash")
print(agent_response)

print("\n" + "="*80)
print("END OF MEMORY RETRIEVAL & AGENT RESPONSE PIPELINE")
print("="*80)


PHASE 5: Final Prompt Building & Agent Response Generation

Test Worker Input: I'm working on welding station 3 today. What should I do if the station overheats?

Final Prompt Built
Prompt length: 2741 characters
Prompt tokens (estimated): 1800

--------------------------------------------------------------------------------
FINAL AGENT RESPONSE (using qwen3.5-flash):
--------------------------------------------------------------------------------
**Action:** Review SOP and instruct based on Site Manager directive.

**Reasoning:**
1.  **Relevant Context:** Memory 1 directly addresses Welding Station 3 overheating. It presents a conflict between Mack (Senior Welder) suggesting a 5% current drop and Sarah (Site Manager) mandating a 15-minute power-down per the new SOP.
2.  **Conflict Resolution:** The memory explicitly documents Sarah overriding Mack's recommendation with "No, follow the SOP. Safety first." This establishes the Site Manager's instruction as the authoritative safety proto

In [11]:
# ============================================================================
# SUMMARY: End-to-End Pipeline Results
# ============================================================================

print("\n" + "="*80)
print("PIPELINE EXECUTION SUMMARY")
print("="*80)

print("\n📊 PHASE 1: Data Parsing & Memory Grouping")
print(f"  ✓ Total exchanges parsed: {len(exchanges)}")
print(f"  ✓ Memory groups created: {len(memories)}")

print("\n🎯 PHASE 2: Memory Scoring (qwen-max)")
print(f"  ✓ Successfully scored: {len(scored_memories)}/{len(memories)} memories")
print(f"  ✓ Scoring criteria: Recency, Importance, Relevance, Experience (each 1-10)")

print("\n📈 PHASE 3: Score Normalization & Filtering")
print(f"  ✓ Normalization applied: min-max scaling to [0, 1] per criterion")
print(f"  ✓ Score range after normalization: [0, 4.0]")
print(f"  ✓ Threshold: 2.0 (adjustable)")
print(f"  ✓ Filtered memories: {len(filtered_memories)}/{len(normalized_memories)}")

print("\n💾 PHASE 4: Context Window Fitting")
print(f"  ✓ Context window: {context_window} tokens")
print(f"  ✓ System prompt reserved: {system_prompt_tokens} tokens")
print(f"  ✓ Available for memories: {context_window - system_prompt_tokens} tokens")
print(f"  ✓ Selected memories: {len(selected_memories)}")
print(f"  ✓ Tokens used: {tokens_used}/{context_window - system_prompt_tokens}")

print("\n🤖 PHASE 5: Agent Response Generation (qwen3.5-flash)")
print(f"  ✓ Model: qwen3.5-flash")
print(f"  ✓ Worker input: '{test_worker_input}'")
print(f"  ✓ Response generated: ✓")

print("\n" + "="*80)
print("TOP RANKED MEMORIES USED IN FINAL RESPONSE:")
print("="*80)
for i, mem in enumerate(selected_memories):
    print(f"\n📌 Memory {i+1} (Score: {mem['final_score']:.3f})")
    print(f"   Time: {mem['timestamp_start']}-{mem['timestamp_end']}")
    print(f"   Speakers: {', '.join(mem['speakers'])}")
    print(f"   Content Preview:")
    for line in mem['full_text'].split('\n')[:3]:
        print(f"     {line}")
    if len(mem['full_text'].split('\n')) > 3:
        print(f"     ...")



PIPELINE EXECUTION SUMMARY

📊 PHASE 1: Data Parsing & Memory Grouping
  ✓ Total exchanges parsed: 21
  ✓ Memory groups created: 5

🎯 PHASE 2: Memory Scoring (qwen-max)
  ✓ Successfully scored: 5/5 memories
  ✓ Scoring criteria: Recency, Importance, Relevance, Experience (each 1-10)

📈 PHASE 3: Score Normalization & Filtering
  ✓ Normalization applied: min-max scaling to [0, 1] per criterion
  ✓ Score range after normalization: [0, 4.0]
  ✓ Threshold: 2.0 (adjustable)
  ✓ Filtered memories: 4/5

💾 PHASE 4: Context Window Fitting
  ✓ Context window: 6000 tokens
  ✓ System prompt reserved: 1500 tokens
  ✓ Available for memories: 4500 tokens
  ✓ Selected memories: 4
  ✓ Tokens used: 1352/4500

🤖 PHASE 5: Agent Response Generation (qwen3.5-flash)
  ✓ Model: qwen3.5-flash
  ✓ Worker input: 'I'm working on welding station 3 today. What should I do if the station overheats?'
  ✓ Response generated: ✓

TOP RANKED MEMORIES USED IN FINAL RESPONSE:

📌 Memory 1 (Score: 2.857)
   Time: 07:02:15-07:

In [12]:
# Compact Summary - Key Statistics Only
print("\n✅ IMPLEMENTATION COMPLETE!")
print("\n📋 Quick Stats:")
print(f"  • Exchanges: {len(exchanges)} → Memories: {len(memories)} → Scored: {len(scored_memories)}")
print(f"  • Filtered (score ≥ 2.0): {len(filtered_memories)} → Selected for context: {len(selected_memories)}")
print(f"  • Memory tokens used: {tokens_used} / {context_window - system_prompt_tokens} available")
print(f"  • Worker Input: '{test_worker_input}'")
print(f"  • Agent Response: Generated successfully via qwen3.5-flash")
print("\n✨ The 5-phase memory retrieval and agent response pipeline is working!")



✅ IMPLEMENTATION COMPLETE!

📋 Quick Stats:
  • Exchanges: 21 → Memories: 5 → Scored: 5
  • Filtered (score ≥ 2.0): 4 → Selected for context: 4
  • Memory tokens used: 1352 / 4500 available
  • Worker Input: 'I'm working on welding station 3 today. What should I do if the station overheats?'
  • Agent Response: Generated successfully via qwen3.5-flash

✨ The 5-phase memory retrieval and agent response pipeline is working!
